# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BarnaliBhowmik/FlyRank_ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### My response

**Method: Logistic Regression**

I will use Logistic Regression as the first ML model for the CTR / engagement
opportunity scoring lane.

The goal is to rank content pages that may have a CTR opportunity for human
review. Logistic Regression is suitable as a simple and interpretable model
because it produces a probability-like score that can be used to rank pages.

I chose it instead of starting with a more complex model because the purpose
of this week is to test whether an ML model provides useful improvement over
the transparent Week-4 rule, not to maximize complexity.

The model will use the Week-3 contract features:

- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_engaged_sessions`
- `scroll_events`

Client and content IDs will not be used as model features.

The model will be evaluated using the same ranking metric as the Week-4
baseline: **Precision@20**.

The main question is:

> Does Logistic Regression identify more true CTR-opportunity pages in its
> top 20 than the Week-4 baseline, when both are evaluated on the same
> held-out data?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
!pip -q install duckdb huggingface_hub pyarrow

import duckdb
import pyarrow.dataset as ds
from huggingface_hub import HfFileSystem
from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

# Connect to Hugging Face
fs = HfFileSystem(token=HF_TOKEN)

print("Hugging Face login successful.")

Hugging Face login successful.


In [5]:
# March 2026 warehouse data
march_path = (
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/"
)

march_files = fs.glob(march_path + "*.parquet")

print("March files found:", len(march_files))
print(march_files)

March files found: 1
['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet']


In [6]:
# Create the March 2026 PyArrow dataset
march_file = march_files[0]
march_uri = "hf://" + march_file

march_dataset = ds.dataset(
    march_uri,
    format="parquet",
    filesystem=fs
)

print("March 2026 PyArrow dataset created.")

March 2026 PyArrow dataset created.


In [7]:
# Connect March data to DuckDB
con = duckdb.connect()

con.register("march_data", march_dataset)

print("March 2026 data connected successfully.")

March 2026 data connected successfully.


In [8]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_data
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [9]:
# Check that the planned Week-5 feature fields exist
# in the March 2026 warehouse data.

planned_features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

# Read the DuckDB schema
schema_columns = con.sql("""
    DESCRIBE march_data
""").df()["column_name"].tolist()

missing_features = [
    feature
    for feature in planned_features
    if feature not in schema_columns
]

print("Planned Week-3 features:")
for feature in planned_features:
    print(" -", feature)

print("\nMissing features:", missing_features)

if missing_features:
    raise ValueError(
        f"Missing required feature columns: {missing_features}"
    )

print("\nAll planned Week-3 features are available.")

Planned Week-3 features:
 - gsc_avg_position
 - ga4_pageviews
 - ga4_sessions
 - ga4_engaged_sessions
 - scroll_events

Missing features: []

All planned Week-3 features are available.


In [10]:
# Check that the planned Week-5 feature fields exist
# in the March 2026 warehouse data.

planned_features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

schema_columns = con.sql("""
    DESCRIBE march_data
""").df()["column_name"].tolist()

missing_features = [
    feature
    for feature in planned_features
    if feature not in schema_columns
]

print("Planned Week-3 features:")
for feature in planned_features:
    print(" -", feature)

print("\nMissing features:", missing_features)

if missing_features:
    raise ValueError(
        f"Missing required feature columns: {missing_features}"
    )

print("\nAll planned Week-3 features are available.")

Planned Week-3 features:
 - gsc_avg_position
 - ga4_pageviews
 - ga4_sessions
 - ga4_engaged_sessions
 - scroll_events

Missing features: []

All planned Week-3 features are available.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### My response

I will use a **grouped train/test split by client**.

The split is done at the `client_hash_id` level so that rows from the same
client do not appear in both training and test data.

This matters because pages belonging to the same client can share similar
measurement patterns. Allowing the same client into both sets could make the
test result look better than it would on unseen clients.

I will use March 2026 as the development dataset, consistent with the Week-3
data contract.

The split will be:

- **80% of clients:** training
- **20% of clients:** held-out test set

The test set will be kept untouched until model evaluation.

The Week-4 baseline and the Logistic Regression model will be evaluated on
the **same held-out test set** and using the same ranking metric,
**Precision@20**.

This is a grouped validation design rather than a random row-level split.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
# Section 2: Build a grouped train/test split by client.
#
# We first create the March page-level dataset using only the
# fields needed for the Week-5 experiment.

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

model_data = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,
        scroll_events,
        gsc_data_available,
        ga4_data_available,
        gsc_clicks,
        gsc_impressions
    FROM march_data
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position > 0
      AND gsc_avg_position <= 20
      AND gsc_impressions > 0
""").df()

print("Rows available for modeling:", len(model_data))
print("Clients:", model_data["client_hash_id"].nunique())
print("Content pages:", model_data["content_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available for modeling: 2539518
Clients: 47
Content pages: 163911


In [13]:
# Create the grouped train/test split.
# No client appears in both sets.

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

train_clients = set(train_data["client_hash_id"])
test_clients = set(test_data["client_hash_id"])

overlap = train_clients.intersection(test_clients)

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))

if overlap:
    raise ValueError("Client leakage detected: train/test clients overlap.")

print("Grouped split verified: no client appears in both sets.")

Training rows: 1870392
Test rows: 669126
Training clients: 37
Test clients: 10
Client overlap: 0
Grouped split verified: no client appears in both sets.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### My response: Target and evaluation

For Logistic Regression, I convert the Week-3 `ctr_gap` proxy into a binary
opportunity label.

A page is labelled **1 (opportunity)** when its `ctr_gap` is above the
median `ctr_gap` calculated from the training data. Otherwise it is labelled
0.

The median threshold is calculated using training data only, so the test set
does not influence target construction.

The Logistic Regression model uses only the five contracted features:

- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_engaged_sessions`
- `scroll_events`

Both the ML model and the Week-4 baseline are evaluated on the same held-out
test pages.

The evaluation metric is **Precision@20** because the practical use case is
to provide a small ranked review queue for human review.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
# Build page-level March data.
# CTR is calculated from observed March GSC clicks and impressions.
# This is used only to construct the Week-3 ctr_gap proxy.

page_data = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(scroll_events) AS scroll_events,

        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS observed_ctr

    FROM march_data
    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position > 0
      AND gsc_avg_position <= 20
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Page-level rows:", len(page_data))
print("Clients:", page_data["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 163911
Clients: 47


In [16]:
# Calculate the same position buckets used by the Week-4 baseline.

def position_bucket(position):
    if position <= 3:
        return "01-03"
    elif position <= 5:
        return "04-05"
    elif position <= 10:
        return "06-10"
    else:
        return "11-20"

page_data["position_bucket"] = (
    page_data["gsc_avg_position"].apply(position_bucket)
)

# Calculate the observed CTR benchmark for each position bucket.
position_benchmark = (
    page_data
    .groupby("position_bucket")
    .apply(
        lambda x: x["gsc_clicks"].sum() / x["gsc_impressions"].sum()
    )
    .to_dict()
)

page_data["benchmark_ctr"] = (
    page_data["position_bucket"].map(position_benchmark)
)

page_data["ctr_gap"] = (
    page_data["benchmark_ctr"] - page_data["observed_ctr"]
).clip(lower=0)

print("CTR-gap proxy created.")
print(page_data["ctr_gap"].describe())

CTR-gap proxy created.
count    163911.000000
mean          0.002265
std           0.001340
min           0.000000
25%           0.001054
50%           0.002927
75%           0.003247
max           0.004045
Name: ctr_gap, dtype: float64


/tmp/ipykernel_447/114405043.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [17]:
# Use the same client-level train/test split created in Section 2.

train_clients = set(train_data["client_hash_id"])
test_clients = set(test_data["client_hash_id"])

train_pages = page_data[
    page_data["client_hash_id"].isin(train_clients)
].copy()

test_pages = page_data[
    page_data["client_hash_id"].isin(test_clients)
].copy()

print("Training pages:", len(train_pages))
print("Test pages:", len(test_pages))

Training pages: 127831
Test pages: 36080


In [18]:
# Learn the opportunity threshold from TRAINING data only.

ctr_gap_threshold = train_pages["ctr_gap"].median()

train_pages["opportunity"] = (
    train_pages["ctr_gap"] > ctr_gap_threshold
).astype(int)

test_pages["opportunity"] = (
    test_pages["ctr_gap"] > ctr_gap_threshold
).astype(int)

print("Training ctr_gap threshold:", round(ctr_gap_threshold, 6))
print(
    "Training opportunity rate:",
    round(train_pages["opportunity"].mean(), 4)
)
print(
    "Test opportunity rate:",
    round(test_pages["opportunity"].mean(), 4)
)

Training ctr_gap threshold: 0.002927
Training opportunity rate: 0.2847
Test opportunity rate: 0.2521


In [19]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

X_train = train_pages[features]
y_train = train_pages["opportunity"]

X_test = test_pages[features]
y_test = test_pages["opportunity"]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

test_pages["ml_score"] = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [20]:
def precision_at_20(df, score_column, label_column="opportunity"):
    ranked = df.sort_values(
        score_column,
        ascending=False
    ).head(20)

    return ranked[label_column].mean()

In [21]:
# ML Precision@20
ml_precision_20 = precision_at_20(
    test_pages,
    "ml_score"
)

# Recreate the Week-4 baseline score on the SAME test pages.
test_pages["baseline_score"] = (
    test_pages["ctr_gap"] *
    np.log1p(test_pages["gsc_impressions"])
)

baseline_precision_20 = precision_at_20(
    test_pages,
    "baseline_score"
)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_precision_20,
        ml_precision_20
    ]
})

comparison

,method,Precision@20
0,Week-4 baseline,1.00
1,Logistic Regression,0.85


In [22]:
if ml_precision_20 > baseline_precision_20:
    result = "Logistic Regression performs better than the Week-4 baseline."
elif ml_precision_20 < baseline_precision_20:
    result = "The Week-4 baseline performs better than Logistic Regression."
else:
    result = "Logistic Regression and the Week-4 baseline have the same Precision@20."

print(result)

The Week-4 baseline performs better than Logistic Regression.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [24]:
# Inspect the model's top 20 recommendations
top20_ml = (
    test_pages
    .sort_values("ml_score", ascending=False)
    .head(20)
    [
        [
            "client_hash_id",
            "content_hash_id",
            "ml_score",
            "opportunity",
            "gsc_avg_position",
            "ga4_pageviews",
            "ga4_sessions",
            "ga4_engaged_sessions",
            "scroll_events",
            "ctr_gap"
        ]
    ]
    .copy()
)

top20_ml

,client_hash_id,content_hash_id,ml_score,opportunity,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,scroll_events,ctr_gap
101432,client_9958f0a7ae1df715,content_b5491e512f7ce8e9,0.447267,0,15.170852,5.0,5.0,0.0,4.0,0.000000
101682,client_9958f0a7ae1df715,content_202277c2683c95b8,0.444863,0,10.357510,22.0,23.0,0.0,8.0,0.000000
157665,client_9958f0a7ae1df715,content_619b0680d37f7c7b,0.441678,0,17.750000,3.0,3.0,0.0,2.0,0.000000
77392,client_f623b01661d4bfe4,content_ee188264af9fbbe1,0.437893,1,20.000000,0.0,0.0,0.0,0.0,0.003247
159892,client_73cda7b4e4f265ea,content_b22b222c99c2d026,0.437893,1,20.000000,NaN,NaN,NaN,NaN,0.003247
159480,client_3f0ce4d44fe94f3d,content_6cc5419a56c4d745,0.437893,1,20.000000,0.0,0.0,0.0,0.0,0.003247
75987,client_f623b01661d4bfe4,content_635b8f0177f72976,0.437893,1,20.000000,0.0,0.0,0.0,0.0,0.003247
159761,client_73cda7b4e4f265ea,content_f8963cb562f65da6,0.437893,1,20.000000,NaN,NaN,NaN,NaN,0.003247
163563,client_f623b01661d4bfe4,content_100219dc312b86bf,0.437893,1,20.000000,0.0,0.0,0.0,0.0,0.003247
77179,client_73cda7b4e4f265ea,content_f692372753033008,0.437893,1,20.000000,NaN,NaN,NaN,NaN,0.003247


In [25]:
false_positives = top20_ml[top20_ml["opportunity"] == 0].copy()

print("Top-20 ML picks:", len(top20_ml))
print("Top-20 false positives:", len(false_positives))

false_positives

Top-20 ML picks: 20
Top-20 false positives: 3


,client_hash_id,content_hash_id,ml_score,opportunity,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,scroll_events,ctr_gap
101432,client_9958f0a7ae1df715,content_b5491e512f7ce8e9,0.447267,0,15.170852,5.0,5.0,0.0,4.0,0.0
101682,client_9958f0a7ae1df715,content_202277c2683c95b8,0.444863,0,10.357510,22.0,23.0,0.0,8.0,0.0
157665,client_9958f0a7ae1df715,content_619b0680d37f7c7b,0.441678,0,17.750000,3.0,3.0,0.0,2.0,0.0


In [26]:
# Logistic Regression coefficients
coefficients = model.named_steps["logistic_regression"].coef_[0]

feature_effects = (
    pd.DataFrame({
        "feature": features,
        "coefficient": coefficients
    })
    .sort_values("coefficient", ascending=False)
    .reset_index(drop=True)
)

feature_effects

,feature,coefficient
0,ga4_sessions,5.497544
1,scroll_events,0.539605
2,gsc_avg_position,0.157239
3,ga4_engaged_sessions,-1.861733
4,ga4_pageviews,-7.131665


In [27]:
print("Features associated with higher modeled opportunity:")
print(
    feature_effects[feature_effects["coefficient"] > 0]
    [["feature", "coefficient"]]
    .to_string(index=False)
)

print("\nFeatures associated with lower modeled opportunity:")
print(
    feature_effects[feature_effects["coefficient"] < 0]
    [["feature", "coefficient"]]
    .to_string(index=False)
)

Features associated with higher modeled opportunity:
         feature  coefficient
    ga4_sessions     5.497544
   scroll_events     0.539605
gsc_avg_position     0.157239

Features associated with lower modeled opportunity:
             feature  coefficient
ga4_engaged_sessions    -1.861733
       ga4_pageviews    -7.131665


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.